# Data extraction

# SinergoX XM

In [33]:
import datetime as dt
import calendar
import pandas as pd
from pydataxm import *
import requests

In [2]:
ObjAPI = pydataxm.ReadDB()
catalog = ObjAPI.get_collections()
print(f"Number of collections: {len(catalog)}")

Number of collections: 193


In [ ]:
start_date = dt.date(2000, 1, 1)
end_date = dt.date(2026, 11, 16)

In [4]:
datasets_proyecto = {
    # Consumo y demanda (Regulada vs No Regulada y Real)
    "demanda_real_sin": ("DemaReal", "Sistema"),
    "demanda_comercial_reg": ("DemaCome", "Sistema"),
    
    # Operación: Generación por tecnología (Hidráulica vs Térmica) y Capacidad
    "generacion_real": ("Gene", "Sistema"),
    "capacidad_efectiva": ("CapEfecNeta", "Sistema"),
    
    # Hidrología: Embalses y Aportes Hídricos
    "volumen_util_embalses": ("VoluUtilDiarEnergia", "Sistema"),
    "aportes_hidricos": ("AporEner", "Sistema"),
    "media_historica_aportes": ("AporEnerMediHist", "Sistema"),
    
    # Impacto Económico: Precios de Bolsa y Escasez
    "precio_bolsa_nacional": ("PrecBolsNaci", "Sistema"),
    "precio_escasez": ("PrecEsca", "Sistema"),
}

In [5]:
for nombre, (metrica, entidad) in datasets_proyecto.items():
    print(f"Extrayendo {nombre} ({metrica})...")
    try:
        df = ObjAPI.request_data(
            coleccion=metrica,
            metrica=entidad,
            start_date=start_date,
            end_date=end_date
        )
        
        if df is not None and not df.empty:
            # Guardar en formato CSV listo para tu proceso ETL
            df.to_csv(f"data/raw/{nombre}.csv", index=False)
            print(f" -> Guardado exitosamente: data/raw/{nombre}.csv ({len(df)} registros)")
        else:
            print(f" -> Sin datos para {nombre} en el rango seleccionado.")
            
    except Exception as e:
        print(f" -> Error al extraer {nombre}: {e}")

Extrayendo demanda_real_sin (DemaReal)...
 -> Guardado exitosamente: data/raw/demanda_real_sin.csv (9814 registros)
Extrayendo demanda_comercial_reg (DemaCome)...
 -> Guardado exitosamente: data/raw/demanda_comercial_reg.csv (9814 registros)
Extrayendo generacion_real (Gene)...
 -> Guardado exitosamente: data/raw/generacion_real.csv (9814 registros)
Extrayendo capacidad_efectiva (CapEfecNeta)...
 -> Error al extraer capacidad_efectiva: index 0 is out of bounds for axis 0 with size 0
Extrayendo volumen_util_embalses (VoluUtilDiarEnergia)...
No existe la métrica VoluUtilDiarEnergia
 -> Sin datos para volumen_util_embalses en el rango seleccionado.
Extrayendo aportes_hidricos (AporEner)...
 -> Guardado exitosamente: data/raw/aportes_hidricos.csv (9816 registros)
Extrayendo media_historica_aportes (AporEnerMediHist)...
 -> Guardado exitosamente: data/raw/media_historica_aportes.csv (9816 registros)
Extrayendo precio_bolsa_nacional (PrecBolsNaci)...
 -> Guardado exitosamente: data/raw/preci

In [7]:
def extract(nombre, metrica, entidad, start_date, end_date):
    try:
        df = ObjAPI.request_data(
            coleccion=metrica,
            metrica=entidad,
            start_date=start_date,
            end_date=end_date
        )
        
        if df is not None and not df.empty:
            # Guardar en formato CSV listo para tu proceso ETL
            df.to_csv(f"data/raw/{nombre}.csv", index=False)
            print(f" -> Guardado exitosamente: data/raw/{nombre}.csv ({len(df)} registros)")
        else:
            print(f" -> Sin datos para {nombre} en el rango seleccionado.")
            
    except Exception as e:
        print(f" -> Error al extraer {nombre}: {e}")

In [27]:
datasets_mensuales = {
    # Volumen útil en Energía (GWh) - Agregado Sistema y por Embalse
    "volumen_util_energia_sistema": ("VoluUtilDiarEner", "Sistema", dt.date(2000, 1, 1), dt.date(2026, 11, 16)),
    "volumen_util_energia_embalse": ("VoluUtilDiarEner", "Embalse", dt.date(2000, 1, 1), dt.date(2026, 11, 16)),
}


In [28]:
def generar_rangos_mensuales(fecha_inicio, fecha_fin):
    actual = dt.date(fecha_inicio.year, fecha_inicio.month, 1)
    while actual <= fecha_fin:
        ultimo_dia = calendar.monthrange(actual.year, actual.month)[1]
        inicio_mes = max(actual, fecha_inicio)
        fin_mes = min(dt.date(actual.year, actual.month, ultimo_dia), fecha_fin)
        yield inicio_mes, fin_mes
        
        # Avanzar al primer día del siguiente mes
        if actual.month == 12:
            actual = dt.date(actual.year + 1, 1, 1)
        else:
            actual = dt.date(actual.year, actual.month + 1, 1)

In [30]:
for nombre, (metrica, entidad, f_inicio, f_fin) in datasets_mensuales.items():
    print(f"\nIniciando descarga mensual para: {nombre} ({metrica} - {entidad})")
    lista_dfs = []
    
    for inicio_mes, fin_mes in generar_rangos_mensuales(f_inicio, f_fin):
        try:
            df_mes = ObjAPI.request_data(
                coleccion=metrica,
                metrica=entidad,
                start_date=inicio_mes,
                end_date=fin_mes
            )
            if df_mes is not None and not df_mes.empty:
                lista_dfs.append(df_mes)
                print(f"  [OK] {inicio_mes} a {fin_mes}: {len(df_mes)} filas")
        except Exception as e:
            print(f"  [AVISO] Sin datos o error en {inicio_mes} a {fin_mes}: {e}")
            
    # 4. Consolidar todos los meses y guardar en CSV
    if lista_dfs:
        df_total = pd.concat(lista_dfs, ignore_index=True)
        archivo_salida = f"data/raw/{nombre}.csv"
        df_total.to_csv(archivo_salida, index=False)
        print(f"--> Exportado exitosamente: {archivo_salida} ({len(df_total)} registros totales)")
    else:
        print(f"--> No se encontraron datos para {nombre} en todo el rango.")


Iniciando descarga mensual para: volumen_util_energia_sistema (VoluUtilDiarEner - Sistema)
  [OK] 2000-01-01 a 2000-01-31: 31 filas
  [OK] 2000-02-01 a 2000-02-29: 29 filas
  [OK] 2000-03-01 a 2000-03-31: 31 filas
  [OK] 2000-04-01 a 2000-04-30: 30 filas
  [OK] 2000-05-01 a 2000-05-31: 31 filas
  [OK] 2000-06-01 a 2000-06-30: 30 filas
  [OK] 2000-07-01 a 2000-07-31: 31 filas
  [OK] 2000-08-01 a 2000-08-31: 31 filas
  [OK] 2000-09-01 a 2000-09-30: 30 filas
  [OK] 2000-10-01 a 2000-10-31: 31 filas
  [OK] 2000-11-01 a 2000-11-30: 30 filas
  [OK] 2000-12-01 a 2000-12-31: 31 filas
  [OK] 2001-01-01 a 2001-01-31: 31 filas
  [OK] 2001-02-01 a 2001-02-28: 28 filas
  [OK] 2001-03-01 a 2001-03-31: 31 filas
  [OK] 2001-04-01 a 2001-04-30: 30 filas
  [OK] 2001-05-01 a 2001-05-31: 31 filas
  [OK] 2001-06-01 a 2001-06-30: 30 filas
  [OK] 2001-07-01 a 2001-07-31: 31 filas
  [OK] 2001-08-01 a 2001-08-31: 31 filas
  [OK] 2001-09-01 a 2001-09-30: 30 filas
  [OK] 2001-10-01 a 2001-10-31: 31 filas
  [OK]

# Climate Prediction Center

In [6]:
url_roni = "https://www.cpc.ncep.noaa.gov/data/indices/RONI.ascii.txt"
url_oni = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"

In [32]:
# 2. Descargar y leer el índice RONI (Relative Oceanic Niño Index)
df_roni = pd.read_csv(url_roni, sep=r"\s+")
df_roni.to_csv("data/raw/cpc_roni_historico.csv", index=False)
print("RONI descargado:", df_roni.shape)
print(df_roni.tail())

# 3. Descargar y leer el índice ONI tradicional (Oceanic Niño Index)
df_oni = pd.read_csv(url_oni, sep=r"\s+")
df_oni.to_csv("data/raw/cpc_oni_historico.csv", index=False)
print("\nONI descargado:", df_oni.shape)
print(df_oni.tail())

RONI descargado: (921, 3)
    SEAS    YR  ANOM
916  AMJ  2026  0.49
917  MJJ  2026  0.97
918  JJA  2026  1.36
919  JAS  2026  1.69
920  ASO  2026  1.97

ONI descargado: (921, 4)
    SEAS    YR  TOTAL  ANOM
916  AMJ  2026  28.74  0.95
917  MJJ  2026  29.02  1.39
918  JJA  2026  29.09  1.80
919  JAS  2026  29.12  2.16
920  ASO  2026  29.23  2.46


# Ideam

In [36]:
DATASET_ID = "cqmv-a99d"
BASE_URL = f"https://www.datos.gov.co/resource/{DATASET_ID}.json"

In [37]:
resp_muestra = requests.get(BASE_URL, params={"$limit": 5})
df_muestra = pd.DataFrame(resp_muestra.json())
print("Columnas disponibles en el dataset del IDEAM:")
print(df_muestra.columns.tolist())
print("\nMuestra de datos:")
print(df_muestra.head(2))

Columnas disponibles en el dataset del IDEAM:
['codigo', 'nombre', 'categoria', 'estado', 'departamento', 'municipio', 'ubicaci_n', 'altitud', 'fecha_instalacion', 'corriente', 'fecha_suspension']

Muestra de datos:
       codigo                                   nombre  \
0  0035057040  PUENTE CARRETERA QDA SALINAS [35057040]   
1    35125020                  VUELTA LARGA [35125020]   

                 categoria      estado departamento     municipio  \
0             Limnimétrica      Activa         Meta      Restrepo   
1  Climatológica Principal  Suspendida         Meta  Puerto López   

                                           ubicaci_n altitud  \
0  {'type': 'Point', 'coordinates': [4.198344444,...     390   
1  {'type': 'Point', 'coordinates': [3.866666667,...     200   

  fecha_instalacion    corriente fecha_suspension  
0        15/10/1978  RÍO SALINAS              NaN  
1        15/03/1987          NaN       15/08/1989  


In [40]:
limite = 50000
offset = 0
bloques = []
parametros_base = {}
print("\nIniciando extracción por bloques desde Datos Abiertos (IDEAM)...")
while True:
    params = {
        **parametros_base,
        "$limit": limite,
        "$offset": offset,
        "$order": ":id" 
    }
    respuesta = requests.get(BASE_URL, params=params)
    if respuesta.status_code != 200:
        print(f"Error HTTP {respuesta.status_code}: {respuesta.text}")
        break
    datos = respuesta.json()
    
    if not datos:
        break
        
    df_bloque = pd.DataFrame(datos)
    bloques.append(df_bloque)
    print(f" -> Descargados {offset + len(df_bloque)} registros...")
    
    offset += limite

# Consolidar y guardar en CSV
if bloques:
    df_ideam_total = pd.concat(bloques, ignore_index=True)
    df_ideam_total.to_csv("data/raw/estaciones_ideam.csv", index=False)
    print(f"\nExportación completa: estaciones_ideam.csv ({len(df_ideam_total)} filas)")
else:
    print("\nNo se descargaron registros.")


Iniciando extracción por bloques desde Datos Abiertos (IDEAM)...
 -> Descargados 649 registros...

Exportación completa: estaciones_ideam.csv (649 filas)


# Alternativa Ideam

Open-Meteo Historical Weather API

In [4]:
import time
import requests
import pandas as pd

departamentos_coords = {
    "ANTIOQUIA": (6.25, -75.56),
    "CUNDINAMARCA": (4.71, -74.07),
    "BOYACÁ": (5.53, -73.36),
    "HUILA": (2.93, -75.28),
    "TOLIMA": (4.43, -75.23),
    "CALDAS": (5.07, -75.52),
    "SANTANDER": (7.12, -73.12),
    "ATLÁNTICO": (10.96, -74.80),
    "BOLÍVAR": (10.39, -75.48),
    "CÓRDOBA": (8.75, -75.88),
    "MAGDALENA": (11.24, -74.20),
    "CESAR": (10.46, -73.25),
}

nombres = list(departamentos_coords.keys())
latitudes = ",".join(str(v[0]) for v in departamentos_coords.values())
longitudes = ",".join(str(v[1]) for v in departamentos_coords.values())

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": latitudes,
    "longitude": longitudes,
    "start_date": "2000-01-01",
    "end_date": "2026-11-15",
    "daily": "precipitation_sum,temperature_2m_mean,temperature_2m_max",
    "timezone": "America/Bogota"
}

resp = requests.get(url, params=params, timeout=60)
resp.raise_for_status()
resultados = resp.json()

dfs = []
for depto, item in zip(nombres, resultados):
    daily = item["daily"]
    dfs.append(pd.DataFrame({
        "fecha": daily["time"],
        "departamento": depto,
        "precipitacion_total_mm": daily["precipitation_sum"],
        "temp_promedio_c": daily["temperature_2m_mean"],
        "temp_maxima_c": daily["temperature_2m_max"]
    }))

df_clima = pd.concat(dfs, ignore_index=True)
df_clima.to_csv("data/raw/clima_colombia_diario_2000_2026.csv", index=False)
print(f"Exportado exitosamente: clima_colombia_diario_2000_2026.csv ({len(df_clima)} filas)")

Exportado exitosamente: clima_colombia_diario_2000_2026.csv (117792 filas)


# UPME


Descargar el archivo en el siguiente https://www.upme.gov.co/simec/planeacion-energetica/proyeccion-de-demanda-v2/

Colocar el archivo en data/raw/manual

In [11]:
import pandas as pd

archivo_upme = "data/raw/manual/Anexo_proyeccion_demanda_2026_2040_ver_Ago2026.xlsx"

df_sin_mensual = pd.read_excel(
    archivo_upme,
    sheet_name="DEE_Mensual(GWh_mes)",
    skiprows=8,
    usecols="B:G"
)
df_sin_mensual.columns = [
    "fecha",
    "demanda_sin_esc_medio_gwh",
    "ic_sup_95_gwh",
    "ic_inf_95_gwh",
    "ic_sup_68_gwh",
    "ic_inf_68_gwh"
]
df_sin_mensual = df_sin_mensual.dropna(subset=["fecha"])
df_sin_mensual["fecha"] = pd.to_datetime(df_sin_mensual["fecha"]).dt.strftime("%Y-%m-%d")
df_sin_mensual["anio"] = pd.to_datetime(df_sin_mensual["fecha"]).dt.year
df_sin_mensual["mes"] = pd.to_datetime(df_sin_mensual["fecha"]).dt.month

df_sin_mensual.to_csv("data/raw/upme_proyeccion_sin_mensual.csv", index=False)
print("Exportado: upme_proyeccion_sin_mensual.csv", df_sin_mensual.shape)
print(df_sin_mensual.head())

# 2. Extraer la Demanda Mensual por Áreas Regionales (Caribe, Oriente, Suroccidente, Antioquia, Nordeste)
df_areas_raw = pd.read_excel(
    archivo_upme,
    sheet_name="MensualSIN_Áreas_EE",
    skiprows=9,
    usecols="B,C,J,Q,X,AE,AL"  # Columna de fecha + Escenario Medio de cada área
)
df_areas_raw.columns = [
    "fecha",
    "caribe_esc_medio_gwh",
    "oriente_esc_medio_gwh",
    "suroccidente_esc_medio_gwh",
    "antioquia_esc_medio_gwh",
    "nordeste_esc_medio_gwh",
    "gce_existentes_esc_medio_gwh"
]
df_areas_raw = df_areas_raw.dropna(subset=["fecha"])
df_areas_raw["fecha"] = pd.to_datetime(df_areas_raw["fecha"]).dt.strftime("%Y-%m-%d")

df_areas_raw.to_csv("data/raw/upme_proyeccion_areas_mensual.csv", index=False)
print("\nExportado: upme_proyeccion_areas_mensual.csv", df_areas_raw.shape)
print(df_areas_raw.head())

Exportado: upme_proyeccion_sin_mensual.csv (177, 8)
        fecha  demanda_sin_esc_medio_gwh  ic_sup_95_gwh  ic_inf_95_gwh  \
0  2026-04-01                6827.757020    6993.487373    6662.431056   
1  2026-05-01                7358.428924    7537.040293    7180.253375   
2  2026-06-01                7060.903968    7232.293504    6889.932629   
3  2026-07-01                7473.356261    7727.766685    7219.818723   
4  2026-08-01                7499.028160    7754.312514    7244.619689   

   ic_sup_68_gwh  ic_inf_68_gwh  anio  mes  
0    6916.969789    6738.751440  2026    4  
1    7454.575552    7262.505588  2026    5  
2    7153.163078    6968.859120  2026    6  
3    7610.260836    7336.898903  2026    7  
4    7636.403019    7362.102054  2026    8  

Exportado: upme_proyeccion_areas_mensual.csv (177, 7)
        fecha  caribe_esc_medio_gwh  oriente_esc_medio_gwh  \
0  2026-04-01           1844.915839            1508.681485   
1  2026-05-01           2034.478450            1607.32